# Milestone 3: the acoustic model, and text to speech

**Goal:** a model that turns Bible text it has never seen into a
mel-spectrogram, played back here with Griffin-Lim. The voice will sound
robotic and buzzy (the trained vocoder in milestone 4 fixes that), but **every
word should be understandable**. See `PLAN.md`, section 4, milestone 3.

**Runtime:** GPU (T4). Needs notebook 02's results on Drive
(`ctc/clips_final.jsonl` and the final recogniser). Training saves checkpoints
to Drive; after a disconnect, rerun the setup cells and the training cell and
it continues.

**Steps:**
1. A quick overfitting check on 10 clips (a few minutes), to prove the code learns.
2. Full training (several hours; the log shows the real speed).
3. Listening comparisons and accuracy: the recogniser from milestone 2 reads
   the synthesised speech, giving a character error rate.
4. Type any text and hear it; read a whole unseen psalm.

**Gate to milestone 4:** unseen dev verses are robotic but every word is
understandable, and the recogniser's error rate on synthesised speech is close
to its error rate on the real recordings.

**Background reading:** Karpathy, "Let's build GPT" (transformer blocks); Ren et
al. 2019 and 2020, FastSpeech and FastSpeech 2.

## Setup

In [ ]:
# Get the code, install it, and connect Google Drive.
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/nicholas-on/TTS.git"
BRANCH = "main"
CODE_DIR = Path("/content/TTS")

def github_token():
    """The repository is private, so cloning needs a GitHub token, stored as a
    Colab secret named GITHUB_TOKEN with "Notebook access" switched on.
    See the README, "Running the proof of concept on Google Colab"."""
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except ImportError:
        return None  # not in Colab
    except Exception as e:  # secret missing, or notebook access not granted
        print(f"No GITHUB_TOKEN available ({type(e).__name__}).")
        return None

def git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(
            f"git {args[0]} failed:\n{message}\n"
            "If this says 'Repository not found' or 'could not read Username', the token is "
            "missing or wrong: add a Colab secret named GITHUB_TOKEN (key icon, left sidebar), "
            "switch on 'Notebook access', and rerun this cell.")

token = github_token()
if not CODE_DIR.exists():
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
    git("clone", "-q", "-b", BRANCH, url, str(CODE_DIR))
else:
    git("-C", str(CODE_DIR), "pull", "-q")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE_DIR)], check=True)
if str(CODE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(CODE_DIR / "src"))

from bibletts.colab import mount_drive, gpu_name, disk_free_gb
from bibletts.config import load_config

mount_drive()
cfg = load_config(CODE_DIR / "configs" / "data.yaml")
DRIVE = Path(cfg["paths"]["drive_root"])   # durable
LOCAL = Path(cfg["paths"]["local_root"])   # fast, wiped when the session ends
RAW = DRIVE / "raw"                        # downloaded originals
for d in (DRIVE, LOCAL, RAW):
    d.mkdir(parents=True, exist_ok=True)
print("GPU:", gpu_name(), "| free local disk:", round(disk_free_gb()), "GB")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from bibletts.audio import MelConfig, MelSpectrogram, load_audio, save_wav
from bibletts.corpus import read_manifest, load_chapter, restore_archive
from bibletts.text import CTCVocab, TTSVocab
from bibletts.ctc.data import compute_chapter_mels, load_mels, ClipDataset
from bibletts.ctc.pipeline import load_segments
from bibletts.ctc.train import load_model as load_recogniser, evaluate as recogniser_eval, read_log
from bibletts.acoustic.train import train_acoustic, recogniser_cer
from bibletts.acoustic.synthesis import synthesise_mel, synthesise_text
from bibletts.viz import plot_mel

ac_cfg = load_config(CODE_DIR / "configs" / "acoustic.yaml")
AC_DIR = DRIVE / "acoustic"
OUT_DIR = AC_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.")

if not (LOCAL / "manifest.jsonl").exists():
    restore_archive(DRIVE / "prepared" / "chapters_v1.tar", LOCAL.parent)
metas = [load_chapter(LOCAL, r["book"], r["chapter"]) for r in read_manifest(LOCAL)]

mel_cfg = MelConfig.from_dict(cfg["audio"])
mel = MelSpectrogram(mel_cfg)
hop_s = mel_cfg.hop_length / mel_cfg.sample_rate
compute_chapter_mels(LOCAL, [m["id"] for m in metas], mel, device=device)
mels = load_mels(LOCAL, [m["id"] for m in metas])

clips = load_segments(DRIVE / "ctc" / "clips_final.jsonl")
by_split = {}
for c in clips:
    by_split.setdefault(c["split"], []).append(c)
print({s: f"{len(v)} clips, {sum(c['end_s'] - c['start_s'] for c in v) / 3600:.2f} h"
       for s, v in by_split.items()})

tts_vocab = TTSVocab(cfg["text"]["alphabet"])
ctc_vocab = CTCVocab(cfg["text"]["alphabet"])
rec_path = next(p for p in [DRIVE / "ctc" / "round2" / "final.pt", DRIVE / "ctc" / "round1" / "final.pt"]
                if p.exists())
recogniser, _ = load_recogniser(rec_path, device)
print("Recogniser:", rec_path)

def griffin_lim(log_mel):
    return mel.griffin_lim(torch.as_tensor(log_mel, dtype=torch.float32),
                           n_iter=ac_cfg["synthesis"]["griffin_lim_iters"])

def play(audio, label=""):
    if label:
        print(label)
    display(Audio(audio, rate=mel_cfg.sample_rate))

## 1. Overfitting check (a few minutes)

Train a fresh model on just 10 clips until it nearly memorises them. If the
code is right, the loss falls close to zero and the synthesised clip sounds
like the Griffin-Lim version of the real one. If it does not, there is a bug,
and it is far cheaper to find it now than after hours of training.

In [ ]:
few = by_split["train"][:ac_cfg["overfit"]["clips"]]
overfit = train_acoustic(few, few, mels, tts_vocab, ac_cfg, AC_DIR / "overfit", hop_s, mel,
                         recogniser=recogniser, ctc_vocab=ctc_vocab,
                         steps=ac_cfg["overfit"]["steps"], device=device)
c = few[0]
real = mels[c["chapter"]][:, c["mel_start"]: c["mel_end"]].astype(np.float32)
print(c["text"])
play(griffin_lim(real), "Real mel-spectrogram through Griffin-Lim:")
play(griffin_lim(synthesise_mel(overfit, tts_vocab, c["text"], device=device)),
     "Overfitted model, from text only:")

## 2. Full training (several hours)

Every 2,000 steps this prints the dev losses and the recogniser's character
error rate on synthesised dev clips, and saves two Griffin-Lim samples to
`acoustic/main/samples/` on Drive. The CER is the number to watch: it should
fall steadily. `epochs` in `configs/acoustic.yaml` sets the length; the
default is a guess, and you can stop early once dev CER stops improving.

In [ ]:
model = train_acoustic(by_split["train"], by_split["dev"], mels, tts_vocab, ac_cfg,
                       AC_DIR / "main", hop_s, mel, recogniser=recogniser, ctc_vocab=ctc_vocab,
                       device=device)

In [ ]:
log = read_log(AC_DIR / "main")
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, key, title in [(axes[0], "mel_loss", "train mel loss"), (axes[1], "dev_mel_loss", "dev mel loss"),
                       (axes[2], "dev_synth_cer", "dev CER on synthesised speech")]:
    points = [(r["step"], r[key]) for r in log if key in r]
    if points:
        ax.plot(*zip(*points), marker="o" if key.startswith("dev") else None)
    ax.set(title=title, xlabel="step")
plt.show()

samples = sorted((AC_DIR / "main" / "samples").glob("*_0.wav"))
for path in samples[:: max(1, len(samples) // 4)][-4:]:
    play(load_audio(path, mel_cfg.sample_rate), f"{path.name}: {by_split['dev'][0]['text']}")

## 3. Listening comparisons

For each unseen dev clip, four versions. Each step down adds one source of error:

1. the real recording;
2. the real mel-spectrogram through Griffin-Lim: the best this playback method can do;
3. the model with the real durations: tests the model's sound;
4. the model from text alone: the full system, including predicted timing.

In [ ]:
for c in by_split["dev"][:3]:
    wav = LOCAL / "chapters" / c["chapter"] / "audio.wav"
    real_mel = mels[c["chapter"]][:, c["mel_start"]: c["mel_end"]].astype(np.float32)
    with torch.no_grad():
        tok = torch.tensor([c["tokens"]], device=device)
        dur = torch.tensor([c["durations"]], device=device)
        with_durations = model(tok, dur)[0][0].float().cpu().numpy()
    from_text = synthesise_mel(model, tts_vocab, c["text"], device=device)
    print("\n" + c["text"])
    play(load_audio(wav, mel_cfg.sample_rate, c["start_s"], c["end_s"] - c["start_s"]), "1. real recording")
    play(griffin_lim(real_mel), "2. real mel, Griffin-Lim")
    play(griffin_lim(with_durations), "3. model, real durations")
    play(griffin_lim(from_text), "4. model, text only")
    fig, axes = plt.subplots(1, 2, figsize=(14, 2.5))
    plot_mel(real_mel, mel_cfg.sample_rate, mel_cfg.hop_length, "real", ax=axes[0])
    plot_mel(from_text, mel_cfg.sample_rate, mel_cfg.hop_length, "text only", ax=axes[1])
    plt.show()

## 4. Accuracy

The milestone-2 recogniser reads each clip twice: the real recording's
mel-spectrogram, and the model's mel-spectrogram synthesised from the text
alone. Character error rate (CER) for each, on chapters never used for
training. The gap between the two columns is the error the TTS model adds.

The recogniser was trained on the same reader, so it may share blind spots
with the TTS model. Your own listening is the final judge.

In [ ]:
import pandas as pd

rows = []
for split in ["dev", "test_same", "test_style", "test_names"]:
    group = by_split.get(split, [])[:150]
    if not group:
        continue
    real = recogniser_eval(recogniser, ClipDataset(group, mels, ctc_vocab), device, max_clips=10**9)
    synth = recogniser_cer(model, recogniser, group, tts_vocab, ctc_vocab, device)
    rows.append({"split": split, "clips": len(group), "CER real": real["cer"],
                 "CER synthesised": synth["cer"], "gap": synth["cer"] - real["cer"]})
    for ref, hyp in synth["examples"][:1]:
        print(f"{split}\n   ref: {ref}\n   hyp: {hyp}")
print(pd.DataFrame(rows).round(3).to_string(index=False))

## 5. Type any text

Uses the same cleaning as training: numbers are spelled out, and punctuation
sets the pauses. Long text is split into sentences. Each result is saved to
`acoustic/outputs/` on Drive.

In [ ]:
TEXT = "The Lord is my shepherd; I shall lack nothing. He makes me lie down in green pastures."

audio, sr = synthesise_text(model, tts_vocab, TEXT, mel, cfg["text"], ac_cfg["synthesis"], device)
save_wav(OUT_DIR / "typed.wav", audio, sr)
play(audio, TEXT)

## 6. Read a whole unseen chapter

Psalm 23 is in the `test_style` split: poetry, a style the model never trained
on, from a chapter it never heard.

In [ ]:
from bibletts.corpus import parse_vpl
import json

sources = json.loads((RAW / "sources.json").read_text())
chapter_text = parse_vpl(sources["vpl"])
BOOK, CHAPTER = "PSA", 23
passage = " ".join(raw for _, raw in chapter_text[(BOOK, CHAPTER)])
print(passage)
audio, sr = synthesise_text(model, tts_vocab, passage, mel, cfg["text"], ac_cfg["synthesis"], device)
save_wav(OUT_DIR / f"{BOOK}_{CHAPTER:03d}.wav", audio, sr)
play(audio)

## Checklist

- [ ] Overfitting check: synthesised clip sounds like the real one
- [ ] Dev CER on synthesised speech fell during training
- [ ] Unseen dev verses: robotic, but every word understandable
- [ ] CER table: synthesised close to real (note the numbers here)
- [ ] Psalm 23 read through without skipped or repeated words

**Next:** milestone 4 trains a vocoder (HiFi-GAN) from scratch to replace
Griffin-Lim, turning the robotic voice into a natural one.